# Privify — Fine-tuning Face Detector

This notebook fine-tunes a YOLOv8n face detector. It is **parametrised by
`DATASET_VERSION`** (set in the configuration cell below):

- **`v0.1`** — Mohamed Traore's Face Detection dataset (Roboflow, close-up
  frontal portraits). Reproduces the original `face-detector-v0.1`.
- **`v0.2`** — a small-face-biased subset of WIDER FACE (built by
  `tools/prepare_wider_face.py`), matching the CCTV deployment scale
  distribution.

Everything except the dataset is held identical across versions (same
`yolov8n.pt`, 50 epochs, imgsz 640, batch 16, and the same Ultralytics-default
optimiser), so the v0.1-vs-v0.2 comparison isolates a single variable: the
training data. The output is a `best.pt` checkpoint to upload as a GitHub
Release asset (`face-detector-<version>`).

**Runtime:** Colab **T4 GPU**, ~20–30 minutes.

## 1. Setup

This notebook requires a **Colab T4 GPU** runtime.
Expected total runtime: **~20–30 minutes** (depending on dataset size and
GPU availability).

In [ ]:
import os

REPO_DIR = "/content/privify"
REPO_URL = "https://github.com/jenz26/privify.git"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, pulling latest changes...")
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}

In [ ]:
import os
from pathlib import Path

MARKER = Path("/tmp/.privify_install_done")

if MARKER.exists():
    print("Dependencies already installed (post-restart). Ready to proceed.")
else:
    print("Installing dependencies (this takes 1-2 minutes)...")
    !pip install -q -r requirements.txt
    MARKER.touch()
    print("Done. Restarting kernel to load new dependencies cleanly...")
    print("After the restart, click 'Runtime → Run all' again.")
    os.kill(os.getpid(), 9)

In [ ]:
import sys
from pathlib import Path

REPO_DIR = "/content/privify"

# Ensure src/ is importable from the repo root.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.training import TrainingConfig, train_face_detector

## 1b. Configuration

Set `DATASET_VERSION` to choose which dataset to fine-tune on. All training
hyperparameters are held identical across versions (scientific control); only
the dataset changes, so any difference in deployment recall is attributable to
the training distribution alone.

In [ ]:
# --- Configuration -------------------------------------------------------
# Switch the whole notebook between dataset versions by changing this one line.
DATASET_VERSION = "v0.2"  # "v0.1" (Roboflow close-ups) | "v0.2" (WIDER small-face subset)

DATASET_CONFIGS = {
    "v0.1": {
        # Roboflow re-export (close-up portraits), published as release `dataset-v0.2`.
        "url": "https://github.com/jenz26/privify/releases/download/dataset-v0.2/face-detection-dataset.zip",
        "yaml_paths": {"train": "train/images", "val": "valid/images", "test": "test/images"},
        "release_notes": (
            "YOLOv8n fine-tuned on Mohamed Traore's Face Detection dataset "
            "(Roboflow Universe, CC BY 4.0). 1558 images (1246 train / 156 valid "
            "/ 156 test), 50 epochs."
        ),
    },
    "v0.2": {
        # WIDER FACE small-face-biased subset, published as release `face-detection-dataset-v0.2`.
        "url": "https://github.com/jenz26/privify/releases/download/face-detection-dataset-v0.2/face-detection-dataset-v0.2.zip",
        "yaml_paths": {"train": "images/train", "val": "images/val", "test": "images/test"},
        "release_notes": (
            "YOLOv8n fine-tuned on a small-face-biased WIDER FACE subset "
            "(3000 images, 2400/300/300 split). Same training config as v0.1; "
            "only the dataset differs."
        ),
    },
}

CFG = DATASET_CONFIGS[DATASET_VERSION]
RUN_NAME = f"face_detector_{DATASET_VERSION}"
DATASET_DIR = Path(f"/content/dataset_{DATASET_VERSION}")
OUTPUT_DIR = Path(f"/content/runs/{RUN_NAME}")
print(f"Configured for {DATASET_VERSION}: run={RUN_NAME}")
print(f"Dataset URL: {CFG['url']}")

## 2. Download dataset

The dataset selected by `DATASET_VERSION` is downloaded from its GitHub Release
asset, extracted, and its `data.yaml` patched to an absolute `path:` for
Ultralytics (which otherwise resolves relative paths against its global
`datasets_dir`). Both versions are single-class (`face`) in YOLOv8 format with
an 80/10/10 train/val/test split.

In [ ]:
import yaml

# Download + extract the selected dataset (idempotent: skip if already present).
if not list(DATASET_DIR.rglob("data.yaml")):
    print(f"Downloading {DATASET_VERSION} dataset from GitHub Releases...")
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    !curl -L "{CFG['url']}" -o {DATASET_DIR}/dataset.zip
    print("Extracting...")
    !cd {DATASET_DIR} && unzip -q dataset.zip && rm dataset.zip
else:
    print(f"Dataset already at {DATASET_DIR}, skipping download.")

# Locate data.yaml regardless of how the archive nests it (v0.1 is flat, the
# v0.2 archive nests everything under a wider_face_subset_v0.2/ directory).
matches = list(DATASET_DIR.rglob("data.yaml"))
if not matches:
    raise FileNotFoundError(f"No data.yaml found under {DATASET_DIR}; verify the Release asset.")
DATA_YAML = matches[0]
DATASET_ROOT = DATA_YAML.parent

# Patch data.yaml: absolute `path:` root + relative subpaths per version.
data_config = yaml.safe_load(DATA_YAML.read_text())
data_config["path"] = str(DATASET_ROOT)
data_config["train"] = CFG["yaml_paths"]["train"]
data_config["val"] = CFG["yaml_paths"]["val"]
data_config["test"] = CFG["yaml_paths"]["test"]

# Fail loudly if the expected split directories are missing.
for split, sub in CFG["yaml_paths"].items():
    if not (DATASET_ROOT / sub).exists():
        raise FileNotFoundError(f"Expected {split} dir {DATASET_ROOT / sub} not found.")

DATA_YAML.write_text(yaml.safe_dump(data_config, default_flow_style=False, sort_keys=False))
print("data.yaml patched:")
print(f"  path:  {data_config['path']}")
print(f"  train: {data_config['train']}")
print(f"  val:   {data_config['val']}")
print(f"  test:  {data_config['test']}")

## 3. Configure and launch training

Training runs for 50 epochs on the T4 GPU.  The configuration is **identical**
to v0.1 (same base model, epochs, image size, batch size, and the
Ultralytics-default optimiser): `train_face_detector` is reused unchanged, and
only `dataset_yaml_path` and `output_dir` vary by version.

In [ ]:
config = TrainingConfig(
    dataset_yaml_path=DATA_YAML,
    output_dir=OUTPUT_DIR,
    base_model="yolov8n.pt",
    epochs=50,
    image_size=640,
    batch_size=16,
    device="cuda",
)
result = train_face_detector(config)

print(f"\nTraining completed in {result.elapsed_seconds / 60:.1f} minutes")
print(f"Best weights: {result.best_weights_path}")
print("Final metrics:")
for key, value in result.final_metrics.items():
    print(f"  {key}: {value:.4f}")

## 3b. Training curves

Parse `results.csv` (written by Ultralytics each epoch) and plot the loss and
metric curves, saved next to the weights for the report. `results.csv` is read
with the standard-library `csv` module (the project stays pandas-free).

In [ ]:
import csv

import matplotlib.pyplot as plt

run_dir = result.best_weights_path.parent.parent  # .../<RUN_NAME>/train
results_csv = run_dir / "results.csv"

# Read results.csv (headers can carry leading spaces -> strip them).
with results_csv.open() as f:
    reader = csv.DictReader(f)
    columns = [c.strip() for c in reader.fieldnames]
    rows = {c: [] for c in columns}
    for record in reader:
        for raw_key, value in record.items():
            rows[raw_key.strip()].append(float(value))

epochs = rows["epoch"]

# Loss curves (train vs val).
plt.figure(figsize=(8, 5))
for col, label in [
    ("train/box_loss", "train box"),
    ("train/cls_loss", "train cls"),
    ("train/dfl_loss", "train dfl"),
    ("val/box_loss", "val box"),
    ("val/cls_loss", "val cls"),
    ("val/dfl_loss", "val dfl"),
]:
    if col in rows:
        plt.plot(epochs, rows[col], label=label)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title(f"{RUN_NAME} — loss curves")
plt.legend()
plt.grid(True, alpha=0.3)
loss_png = run_dir / "loss_curves.png"
plt.savefig(loss_png, dpi=120, bbox_inches="tight")
plt.show()

# Metric curves (mAP).
plt.figure(figsize=(8, 5))
for col, label in [
    ("metrics/mAP50(B)", "mAP@50"),
    ("metrics/mAP50-95(B)", "mAP@50-95"),
]:
    if col in rows:
        plt.plot(epochs, rows[col], label=label)
plt.xlabel("epoch")
plt.ylabel("mAP")
plt.title(f"{RUN_NAME} — metric curves")
plt.legend()
plt.grid(True, alpha=0.3)
metrics_png = run_dir / "metrics_curves.png"
plt.savefig(metrics_png, dpi=120, bbox_inches="tight")
plt.show()

print(f"Saved {loss_png}")
print(f"Saved {metrics_png}")

## 4. Download trained weights

Download `best.pt` and the training-curve figures. The cell below also prints
the exact, **version-aware** `gh release create` command (tagged
`face-detector-<DATASET_VERSION>`) to publish the weights as a GitHub Release
asset — run it from a machine with the `gh` CLI authenticated.

In [ ]:
from google.colab import files

# best.pt plus the training-curve figures for the report.
for path in [result.best_weights_path, loss_png, metrics_png]:
    files.download(str(path))

# Version-aware release command (so v0.1 weights never land on the v0.2 tag).
notes = CFG["release_notes"]
print("\nPublish the weights with the gh CLI:\n")
print(f"gh release create face-detector-{DATASET_VERSION} {result.best_weights_path.name} \\")
print(f'    --title "Face detector {DATASET_VERSION}" \\')
print(f'    --notes "{notes}"')

## Next steps

- Upload `best.pt` as a GitHub Release asset tagged `face-detector-<version>`
- Add the v0.2 weights to the OOD evaluation (`notebooks/ood_evaluation.ipynb`)
  as a third column alongside v0.1 and the COCO baseline (Phase 4.3)
- Update the ADR in `docs/decisions.md` with the measured v0.1-vs-v0.2 recall